# LoRA training (diffusion integration)

Experimental path: `yggdrasill.integrations.diffusers.training` — not the minimal stable core.

**Entry points:** `train_diffusion_lora(config=...)`, or `train_sd15_lora` / `train_sdxl_lora` / `train_flux_lora`.

**Rule:** `TrainingConfig.family` must match the base model (`"sd15"` for SD1.5, `"sdxl"` for SDXL, `"flux"` for FLUX). See `documentation/API_SURFACE.md` and `documentation/DIFFUSION_STATUS.md`.

## Train SDXL LoRA (text2img)

### Prerequisites

- **Packages:** `torch`, `diffusers`, `peft`, `torchvision`, `Pillow`; for Hugging Face datasets also `datasets`.
- **Data:** folder of images; for plain `text2img` each image needs a sibling `.txt` caption, **or** `metadata.jsonl` / `metadata.csv` with `image` / `caption` columns (names configurable on `TrainingConfig`).
- **VRAM:** SDXL + text-encoder LoRA is heavy; start with `train_text_encoder=False`, `resolution=512`, `batch_size=1` if needed.

### Tasks

- SD1.5: `text2img`, `img2img`, `inpaint`
- SDXL: `text2img`, `img2img`, `inpaint`, `refiner`
- FLUX: `text2img`

For `img2img` / `inpaint` / `refiner`, use a manifest with the extra columns your task needs (`init_image`, masks, `prompt_2`, aesthetic score, etc.).

In [ ]:
import os
from pathlib import Path

from yggdrasill.integrations.diffusers.training import TrainingConfig, train_diffusion_lora

# Paths: set env vars or edit defaults below.
_root = Path.cwd()
DATA_DIR = os.environ.get("YGG_LORA_DATA_DIR", str(_root / "data" / "lora_train"))
OUTPUT_LORA = os.environ.get("YGG_LORA_OUTPUT", str(_root / "artifacts" / "my_sdxl_lora.safetensors"))
Path(OUTPUT_LORA).parent.mkdir(parents=True, exist_ok=True)

PRETRAINED_SDXL = os.environ.get(
    "YGG_PRETRAINED_SDXL",
    "stabilityai/stable-diffusion-xl-base-1.0",
)

config = TrainingConfig(
    pretrained_model_name_or_path=PRETRAINED_SDXL,
    data_dir=DATA_DIR,
    output_path=OUTPUT_LORA,
    family="sdxl",
    task="text2img",
    mixed_precision="fp16",
    resolution=1024,
    num_epochs=1,
    batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    max_train_steps=2000,
    lora_rank=16,
    lora_alpha=16,
    logging_steps=10,
    checkpoint_every_n_steps=500,
    # Set True if you have headroom; SDXL TE LoRA is memory-intensive.
    train_text_encoder=False,
    train_text_encoder_2=False,
)

# Equivalent shortcut:
# from yggdrasill.integrations.diffusers.training import train_sdxl_lora
# result = train_sdxl_lora(data_dir=DATA_DIR, pretrained=PRETRAINED_SDXL, output_path=OUTPUT_LORA, ...)

result = train_diffusion_lora(config=config)
result.output_path

FileNotFoundError: Couldn't find any data file at /workspace/YggDrasill/examples/data/lora_train.

## Inference: compare base vs LoRA

Uses the same `OUTPUT_LORA` path as training. Pick `device` automatically when CUDA is available.

In [ ]:
import os
from pathlib import Path

import torch
from IPython.display import display

from yggdrasill import DiffusionGraphBuilder

# Same defaults as training cell (safe if you run this cell alone).
_root = Path.cwd()
OUTPUT_LORA = os.environ.get("YGG_LORA_OUTPUT", str(_root / "artifacts" / "my_sdxl_lora.safetensors"))

device = "cuda" if torch.cuda.is_available() else "cpu"
lora_path = str(Path(OUTPUT_LORA).resolve())

base = DiffusionGraphBuilder.from_template("sdxl_text2img", device=device)
with_lora = DiffusionGraphBuilder.from_template("sdxl_text2img", device=device)
with_lora.add_component("lora", "sdxl.lora", pretrained=lora_path)

common_kwargs = dict(
    prompt="a photo of a red sports car on a coastal road, golden hour",
    negative_prompt="blurry, low quality",
    num_inference_steps=30,
    guidance_scale=7.0,
    num_images_per_prompt=1,
    seed=42,
    width=1024,
    height=1024,
)

base_img = base.run(**common_kwargs).images[0]
lora_img = with_lora.run(**common_kwargs, lora_conditioning_scale=1.0).images[0]

print("base vs LoRA")
display(base_img)
display(lora_img)